In [7]:
import pandas as pd
import numpy as np

In [3]:
cluster_means_real = pd.read_csv("cluster_means_real.csv", index_col=0)

profile_table = pd.read_csv(
    "profile_table.csv",
    header=[0,1],
    index_col=0
)

cluster_means_real, profile_table

(                     Income  Customer Lifetime Value  Tenure_Years  \
 merged_labels                                                        
 0              36800.694896             14873.856584      2.022434   
 1              37766.295064              6088.021802      3.579614   
 2              37745.929617              6791.045903      0.965109   
 
                Companions_ratio  PointsAccumulated  DollarCostPointsRedeemed  \
 merged_labels                                                                  
 0                      5.437076       28221.461015                 84.672126   
 1                      7.263742       36767.343185                110.361738   
 2                      1.512713        8581.270118                 23.681151   
 
                Flights_Per_Year  CLV_per_Year  
 merged_labels                                  
 0                     48.194990   8876.392337  
 1                     62.504685   1845.331671  
 2                     14.373371   1107.

In [6]:
n_customers = profile_table[('cluster_info','n_customers')]
cluster_means_real['customers'] = n_customers.values
cluster_means_real

,Income,Customer Lifetime Value,Tenure_Years,Companions_ratio,PointsAccumulated,DollarCostPointsRedeemed,Flights_Per_Year,CLV_per_Year,customers
merged_labels,,,,,,,,,
0,36800.694896,14873.856584,2.022434,5.437076,28221.461015,84.672126,48.194990,8876.392337,1783
1,37766.295064,6088.021802,3.579614,7.263742,36767.343185,110.361738,62.504685,1845.331671,9320
2,37745.929617,6791.045903,0.965109,1.512713,8581.270118,23.681151,14.373371,1107.672263,4987


In [8]:
cluster_means_real["clv_growth_ratio"] = (
    cluster_means_real["CLV_per_Year"] / cluster_means_real["Customer Lifetime Value"]
)
cluster_means_real[["Customer Lifetime Value", "CLV_per_Year", "clv_growth_ratio"]]

,Customer Lifetime Value,CLV_per_Year,clv_growth_ratio
merged_labels,,,
0,14873.856584,8876.392337,0.596778
1,6088.021802,1845.331671,0.303109
2,6791.045903,1107.672263,0.163108


In [23]:
segment_costs = {
    0: 900_000,   # High-Value Core → premium, high-touch initiatives
    1: 1_800_000, # Loyal & Engaged → scalable platform & personalization
    2: 300_000    # Low Engagement → low-cost automated activation
}

In [24]:
cluster_means_real["cost_allocated"] = cluster_means_real.index.map(segment_costs)
cluster_means_real[["customers", "cost_allocated"]]

,customers,cost_allocated
merged_labels,,
0,1783,900000
1,9320,1800000
2,4987,300000


In [25]:
scenarios = {
    "Conservative": {0: 0.15, 1: 0.20, 2: 0.10},
    "Base case":    {0: 0.20, 1: 0.30, 2: 0.15},
    "Optimistic":   {0: 0.30, 1: 0.40, 2: 0.25},
}

In [26]:
rows = []

for scen, weights in scenarios.items():
    
    df = cluster_means_real.copy()
    
    # Segment-specific uplift derived from growth
    df["uplift"] = df.index.map(
        lambda k: df.loc[k, "clv_growth_ratio"] * weights[k]
    )
    
    # Annual incremental value per segment
    df["annual_incremental_value"] = (
        df["CLV_per_Year"] * df["uplift"] * df["customers"]
    )
    
    # ROI per segment
    df["segment_roi"] = (
        df["annual_incremental_value"] - df["cost_allocated"]
    ) / df["cost_allocated"]
    
    # Store results
    for k in df.index:
        rows.append({
            "Scenario": scen,
            "Cluster": k,
            "Uplift": round(df.loc[k, "uplift"], 3),
            "Annual_Incremental_Value": round(df.loc[k, "annual_incremental_value"], 0),
            "Allocated_Cost": round(df.loc[k, "cost_allocated"], 0),
            "Segment_ROI": round(df.loc[k, "segment_roi"], 2)
        })

segment_roi_table = pd.DataFrame(rows)
segment_roi_table

,Scenario,Cluster,Uplift,Annual_Incremental_Value,Allocated_Cost,Segment_ROI
0,Conservative,0,0.090,1416746.0,900000,0.57
1,Conservative,1,0.061,1042602.0,1800000,-0.42
2,Conservative,2,0.016,90100.0,300000,-0.70
3,Base case,0,0.119,1888995.0,900000,1.10
4,Base case,1,0.091,1563903.0,1800000,-0.13
5,Base case,2,0.024,135150.0,300000,-0.55
6,Optimistic,0,0.179,2833492.0,900000,2.15
7,Optimistic,1,0.121,2085204.0,1800000,0.16
8,Optimistic,2,0.041,225250.0,300000,-0.25


In [27]:
timeline_weights = {
    0: {"short": 0.10, "medium": 0.30, "long": 0.60},
    1: {"short": 0.20, "medium": 0.50, "long": 0.30},
    2: {"short": 0.50, "medium": 0.40, "long": 0.10},
}

timeline_rows = []

for k in df.index:
    val = df.loc[k, "annual_incremental_value"]
    timeline_rows.append({
        "Cluster": k,
        "Short_term_0_3m (€)": round(val * timeline_weights[k]["short"], 0),
        "Medium_term_3_9m (€)": round(val * timeline_weights[k]["medium"], 0),
        "Long_term_9_18m (€)": round(val * timeline_weights[k]["long"], 0),
        "Total_Annual (€)": round(val, 0)
    })

timeline_table = pd.DataFrame(timeline_rows)
timeline_table

,Cluster,Short_term_0_3m (€),Medium_term_3_9m (€),Long_term_9_18m (€),Total_Annual (€)
0,0,283349.0,850048.0,1700095.0,2833492.0
1,1,417041.0,1042602.0,625561.0,2085204.0
2,2,112625.0,90100.0,22525.0,225250.0
